# Validation Tests — 6-DOF Ballistic Trajectory Simulator

Three automated validation tests that verify the physics engine is correct.

| Test | Description | Criterion |
|------|-------------|----------|
| 1 | Vacuum trajectory range | < 1 % error vs analytical $R = v_0^2/g$ |
| 2 | Energy conservation (vacuum) | < 0.1 % deviation |
| 3 | ISA sea-level properties | exact / 0.1 % |

### Physics Background

In a vacuum (no air resistance), a projectile launched at angle $\theta$ with speed $v_0$ follows a parabolic trajectory. The maximum range is achieved at $\theta = 45°$ and equals:

$$R_{\text{vacuum}} = \frac{v_0^2}{g}$$

This is the most fundamental check: if our integrator and EOM reproduce this exactly, the gravitational dynamics are correct.

Similarly, total mechanical energy $E = \frac{1}{2}mv^2 + mgh$ must be conserved in the absence of dissipative forces.

In [ ]:
import sys, math
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, '..')

from src.atmosphere import ISAAtmosphere
from src.aerodynamics import AeroModel
from src.integrator import TrajectoryIntegrator

---
## Test 1 — Vacuum Trajectory Check

Set $C_D = C_L = C_M = 0$, all moment coefficients to zero, and no wind.

Launch at 45° elevation with $v_0 = 300$ m/s. Compare simulated range against analytical.

In [ ]:
GRAVITY_MS2 = 9.80665
V0_MS = 300.0
ELEVATION_DEG = 45.0

# Zero-aero model (no forces, no moments)
aero_vacuum = AeroModel(
    Cd=0.0, Cl=0.0, Cm=0.0,
    reference_area_m2=0.01,
    reference_diameter_m=0.1,
    Cmq=0.0, Clp=0.0, Cnpa=0.0,
)
atm = ISAAtmosphere()
integrator = TrajectoryIntegrator()

elev_rad = math.radians(ELEVATION_DEG)
initial_state = np.array([
    0.0, 0.0, 0.0,
    V0_MS * math.cos(elev_rad), 0.0, V0_MS * math.sin(elev_rad),
    1.0, 0.0, 0.0, 0.0,
    0.0, 0.0, 0.0,
])

inertia = np.diag([0.001, 0.001, 0.001])

result = integrator.integrate(
    initial_state=initial_state,
    t_span=(0.0, 200.0),
    dt=0.01,
    mass_kg=1.0,
    inertia_tensor=inertia,
    aero_model=aero_vacuum,
    atmosphere=atm,
    wind_vector_ms=np.zeros(3),
)

simulated_range_m = result.state[-1, 0]
analytical_range_m = V0_MS**2 / GRAVITY_MS2
error_pct = abs(simulated_range_m - analytical_range_m) / analytical_range_m * 100

print(f'Simulated range : {simulated_range_m:,.2f} m')
print(f'Analytical range: {analytical_range_m:,.2f} m')
print(f'Relative error  : {error_pct:.4f} %')
print()
if error_pct < 1.0:
    print('TEST 1: ✅ PASS')
else:
    print('TEST 1: ❌ FAIL')

---
## Test 2 — Energy Conservation (Vacuum)

Total mechanical energy:

$$E = \frac{1}{2}mv^2 + mgz$$

must be conserved to within 0.1% throughout the vacuum flight.

In [ ]:
mass_kg = 1.0
z = result.state[:, 2]
speed = result.speed_ms

KE_J = 0.5 * mass_kg * speed**2
PE_J = mass_kg * GRAVITY_MS2 * z
E_total_J = KE_J + PE_J

E0 = E_total_J[0]
max_deviation_pct = np.max(np.abs(E_total_J - E0)) / abs(E0) * 100

# Plot energy components
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(result.time_s, KE_J, label='Kinetic Energy', color='#ff7f0e')
ax.plot(result.time_s, PE_J, label='Potential Energy', color='#2ca02c')
ax.plot(result.time_s, E_total_J, label='Total Energy', color='#1f77b4', linewidth=2)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Energy [J]')
ax.set_title('Energy Conservation Check (Vacuum Flight)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Initial energy     : {E0:,.2f} J')
print(f'Max deviation      : {max_deviation_pct:.6f} %')
print()
if max_deviation_pct < 0.1:
    print('TEST 2: ✅ PASS')
else:
    print('TEST 2: ❌ FAIL')

---
## Test 3 — ISA Sea-Level Properties

The U.S. Standard Atmosphere 1976 defines the following sea-level values:

| Property | Value | Unit |
|----------|-------|------|
| Temperature | 288.15 | K |
| Pressure | 101,325 | Pa |
| Density | 1.225 | kg/m³ |
| Speed of sound | 340.29 | m/s |

In [ ]:
props = ISAAtmosphere().get_properties(0.0)

temp_ok = (props['temperature_K'] == 288.15)
density_error_pct = abs(props['density_kg_m3'] - 1.225) / 1.225 * 100
density_ok = (density_error_pct < 0.1)

print(f"Temperature    : {props['temperature_K']} K       (expected 288.15)   {'✅' if temp_ok else '❌'}")
print(f"Density        : {props['density_kg_m3']:.6f} kg/m³  (expected 1.225)    err {density_error_pct:.4f}%  {'✅' if density_ok else '❌'}")
print(f"Pressure       : {props['pressure_Pa']:.2f} Pa      (expected 101325)")
print(f"Speed of sound : {props['speed_of_sound_m_s']:.2f} m/s     (expected 340.29)")
print()
if temp_ok and density_ok:
    print('TEST 3: ✅ PASS')
else:
    print('TEST 3: ❌ FAIL')

---
## Summary

| Test | Description | Criterion | Result |
|------|-------------|-----------|--------|
| 1 | Vacuum trajectory range | < 1% error | Run above |
| 2 | Energy conservation | < 0.1% deviation | Run above |
| 3 | ISA sea-level values | exact / 0.1% | Run above |

All three tests should show **✅ PASS**. If any test fails, investigate the corresponding module.